# Load Dataset

In [3]:
from beir.datasets.data_loader import GenericDataLoader

data_path = "scifact"

corpus, queries, qrels = GenericDataLoader(
    data_path
).load(split="test")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 99684.87it/s]


In [5]:
import pandas as pd

corpus_df = pd.DataFrame.from_dict(
    corpus,
    orient="index"   
).reset_index()

In [6]:
corpus_df

,index,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [7]:
documents = [
    row['title'] + ". " + row['text'] 
    for _, row in corpus_df.iterrows()
]

# Embedding Model

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2416.60it/s]


In [ ]:
import time

start_time = time.time()

corpus_embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

end_time = time.time()

print(f"Encoding took {end_time - start_time:.2f} seconds")
print(document_embeddings.shape)

In [11]:
corpus_df['embedding'] = list(corpus_embeddings)

# Faiss

In [12]:
import faiss

In [13]:
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings.astype('float32'))

In [14]:
index.ntotal

5183

In [15]:
queries

{'1': '0-dimensional biomaterials show inductive properties.',
 '3': '1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.',
 '5': '1/2000 in UK have abnormal PrP positivity.',
 '13': '5% of perinatal mortality is due to low birth weight.',
 '36': 'A deficiency of vitamin B12 increases blood levels of homocysteine.',
 '42': 'A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.',
 '48': 'A total of 1,000 people in the UK are asymptomatic carriers of vCJD infection.',
 '49': 'ADAR1 binds to Dicer to cleave pre-miRNA.',
 '50': 'AIRE is expressed in some skin tumors.',
 '51': 'ALDH1 expression is associated with better breast cancer outcomes.',
 '53': 'ALDH1 expression is associated with poorer prognosis in breast cancer.',
 '54': 'AMP-activated protein kinase (AMPK) activation increases inflammation-related fibrosis in the lung

In [16]:
qrels

{'1': {'31715818': 1},
 '3': {'14717500': 1},
 '5': {'13734012': 1},
 '13': {'1606628': 1},
 '36': {'5152028': 1, '11705328': 1},
 '42': {'18174210': 1},
 '48': {'13734012': 1},
 '49': {'5953485': 1},
 '50': {'12580014': 1},
 '51': {'45638119': 1},
 '53': {'45638119': 1},
 '54': {'49556906': 1},
 '56': {'4709641': 1},
 '57': {'4709641': 1},
 '70': {'5956380': 1, '4414547': 1},
 '72': {'6076903': 1},
 '75': {'4387784': 1},
 '94': {'1215116': 1},
 '99': {'18810195': 1},
 '100': {'4381486': 1},
 '113': {'6157837': 1},
 '115': {'33872649': 1},
 '118': {'6372244': 1},
 '124': {'4883040': 1},
 '127': {'21598000': 1},
 '128': {'8290953': 1},
 '129': {'27768226': 1},
 '130': {'27768226': 1},
 '132': {'7975937': 1},
 '133': {'38485364': 1,
  '6969753': 1,
  '17934082': 1,
  '16280642': 1,
  '12640810': 1},
 '137': {'26016929': 1},
 '141': {'6955746': 1, '14437255': 1},
 '142': {'10582939': 1},
 '143': {'10582939': 1},
 '146': {'10582939': 1},
 '148': {'1084345': 1},
 '163': {'18872233': 1},
 '1

In [17]:
queries_df = (pd.DataFrame.from_dict(queries, orient="index")
              .reset_index()
              .rename(columns={'index': 'query_id', 0: 'query_text'}))

queries_df

,query_id,query_text
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [18]:
qrels_list = []
for query_id, docs in qrels.items():
    for doc_id, relevance in docs.items():
        qrels_list.append({
            'query_id': query_id,
            'doc_id': doc_id,
            'relevance': relevance
        })

qrels_df = pd.DataFrame(qrels_list)
qrels_df.head(10)

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
5,36,11705328,1
6,42,18174210,1
7,48,13734012,1
8,49,5953485,1
9,50,12580014,1


In [19]:
qrels_df['relevance'].value_counts()

relevance
1    339
Name: count, dtype: int64

# Test with querie number 1

In [20]:
queries_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   query_id    300 non-null    str  
 1   query_text  300 non-null    str  
dtypes: str(2)
memory usage: 32.2 KB


In [21]:
query = queries_df.loc[queries_df['query_id'] == '70', 'query_text'].iloc[0]

In [23]:
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

In [24]:
scores, indices = index.search(
    query_embedding,
    k=10
)

In [25]:
for rank, idx in enumerate(indices[0], start=1):

    print(f"\nRank {rank}")
    print("Score:", scores[0][rank - 1])
    print("ID:", corpus_df.iloc[idx]["index"])


Rank 1
Score: 0.78618187
ID: 4414547

Rank 2
Score: 0.7666184
ID: 11903247

Rank 3
Score: 0.76654863
ID: 5956380

Rank 4
Score: 0.7640258
ID: 31624828

Rank 5
Score: 0.75287116
ID: 7225911

Rank 6
Score: 0.750954
ID: 21557055

Rank 7
Score: 0.747499
ID: 9483851

Rank 8
Score: 0.73814404
ID: 12240507

Rank 9
Score: 0.7373251
ID: 2582169

Rank 10
Score: 0.736439
ID: 36464673


In [26]:
qrels_df.loc[qrels_df['query_id'] == '70', 'doc_id']

15    5956380
16    4414547
Name: doc_id, dtype: str

# Evaluation Using Ranx

In [ ]:
run_dict = {}

for i, row in queries_df.iterrows():
    query_id = row['query_id']  
    query_txt = row['query_text']
    
    # Get embeddings and search
    query_embedding = embedding_model.encode(
        [query_txt],
        normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding, k=10)
    
    doc_scores = {}
    for rank, idx in enumerate(indices[0], start=1):  # indices[0] because only 1 query
        doc_id = corpus_df.iloc[idx]["index"]
        score = scores[0][rank - 1]
        doc_scores[doc_id] = float(score)  
    
    # Add to run_dict
    run_dict[query_id] = doc_scores